In [16]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import numpy as np
import psutil

torch.manual_seed(42)
np.random.seed(42)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

feat = torch.load('../data/canwell_features.pt')
x        = feat['x']
y        = feat['y']
is_slope = feat['is_slope']
is_basin = feat['is_basin']
has_diff = feat['has_diff']
edge_index = torch.load('../data/canwell_edgidx.pt')

print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 29.5%


In [3]:
from sklearn.preprocessing import StandardScaler

# normalize node features
x_np = x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

In [4]:
# is_masked flag
is_masked = torch.zeros(x.shape[0], dtype=torch.float)
slope_with_diff = (is_slope & has_diff).nonzero(as_tuple=True)[0]

In [5]:
# use same split as before for fair comparison
perm = torch.randperm(len(slope_with_diff), generator=torch.Generator().manual_seed(42))
n = len(slope_with_diff)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

In [6]:
train_idx = slope_with_diff[perm[:train_end]]
val_idx   = slope_with_diff[perm[train_end:val_end]]
test_idx  = slope_with_diff[perm[val_end:]]

In [7]:
is_masked[train_idx] = 1.0
is_masked[val_idx]   = 1.0
is_masked[test_idx]  = 1.0

In [8]:
# normalize y
y_mean = y[train_idx].mean().item()
y_std  = y[train_idx].std().item()
y_norm = (y - y_mean) / y_std

In [9]:
# stack features — diff_context zeroed out (ablation)
diff_context = torch.zeros(x.shape[0], dtype=torch.float)  # <-- key change

x_final = torch.cat([
    torch.tensor(x_scaled, dtype=torch.float),
    is_masked.unsqueeze(1),
    diff_context.unsqueeze(1)   # all zeros, no basin signal
], dim=1)

data = Data(x=x_final, edge_index=edge_index, y=y_norm)
data.is_slope = is_slope
data.is_basin = is_basin
data.has_diff = has_diff

print(f"Feature matrix shape: {data.x.shape}")
print(f"Basin signal: ZEROED OUT (ablation)")
print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

Feature matrix shape: torch.Size([4792230, 6])
Basin signal: ZEROED OUT (ablation)
Train: 1,986,717 | Val: 425,725 | Test: 425,726
RAM: 21.9%


In [10]:
class CanwellSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super(CanwellSAGE, self).__init__()
        
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        
        self.lin1 = Linear(hidden_channels, hidden_channels // 2)
        self.lin2 = Linear(hidden_channels // 2, 1)
    
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.lin1(x))
        x = self.lin2(x)
        return x.squeeze(1)

model = CanwellSAGE(in_channels=6, hidden_channels=64).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

train_loader = NeighborLoader(data, num_neighbors=[10,10,10], batch_size=512,
                               input_nodes=train_idx, shuffle=True)
val_loader   = NeighborLoader(data, num_neighbors=[10,10,10], batch_size=512,
                               input_nodes=val_idx, shuffle=False)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


Model parameters: 19,457
RAM: 30.5%


In [11]:
# training functions
def train():
    model.train()
    total_loss = 0
    count = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        out = out[:batch.batch_size]
        target = batch.y[:batch.batch_size]
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        count += batch.batch_size
    return total_loss / count

def validate():
    model.eval()
    total_loss = 0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            out = out[:batch.batch_size]
            target = batch.y[:batch.batch_size]
            loss = criterion(out, target)
            total_loss += loss.item() * batch.batch_size
            count += batch.batch_size
    return total_loss / count

In [13]:
# training loop
best_val_loss = float('inf')
best_model = None
patience = 10
no_improve = 0
max_epochs = 100
train_losses = []
val_losses = []

for epoch in range(max_epochs):
    train_loss = train()
    val_loss   = validate()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | RAM: {psutil.virtual_memory().percent:.1f}%")

    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_model = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        print(f"  ✓ New best: {best_val_loss:.4f} at epoch {epoch}")
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        print(f"Best epoch: {best_epoch}, Best val MSE: {best_val_loss:.4f}")
        model.load_state_dict(best_model)
        break

print("\nTraining complete.")

Epoch 000 | Train MSE: 0.3921 | Val MSE: 0.3499 | RAM: 34.4%
  ✓ New best: 0.3499 at epoch 0
  ✓ New best: 0.3431 at epoch 1
  ✓ New best: 0.3206 at epoch 2
  ✓ New best: 0.3098 at epoch 3
Epoch 005 | Train MSE: 0.3114 | Val MSE: 0.3014 | RAM: 34.5%
  ✓ New best: 0.3014 at epoch 5
  ✓ New best: 0.2978 at epoch 6
  ✓ New best: 0.2945 at epoch 7
  ✓ New best: 0.2931 at epoch 8
  ✓ New best: 0.2878 at epoch 9
Epoch 010 | Train MSE: 0.2962 | Val MSE: 0.2860 | RAM: 34.3%
  ✓ New best: 0.2860 at epoch 10
  ✓ New best: 0.2806 at epoch 11
  ✓ New best: 0.2794 at epoch 13
  ✓ New best: 0.2781 at epoch 14
Epoch 015 | Train MSE: 0.2879 | Val MSE: 0.2741 | RAM: 34.4%
  ✓ New best: 0.2741 at epoch 15
  ✓ New best: 0.2717 at epoch 17
  ✓ New best: 0.2696 at epoch 19
Epoch 020 | Train MSE: 0.2811 | Val MSE: 0.2677 | RAM: 34.4%
  ✓ New best: 0.2677 at epoch 20
  ✓ New best: 0.2659 at epoch 21
  ✓ New best: 0.2651 at epoch 23
Epoch 025 | Train MSE: 0.2766 | Val MSE: 0.2632 | RAM: 34.5%
  ✓ New best: 0.

In [18]:
# run this immediately after training completes
torch.save({
    'best_model': best_model,
    'best_val_loss': best_val_loss,
    'best_epoch': best_epoch,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'y_mean': y_mean,
    'y_std': y_std,
    'test_idx': test_idx,
    'train_idx': train_idx,
    'val_idx': val_idx,
}, 'canwell_sage_nobasin.pt')
print("Saved!")

Saved!


In [39]:
torch.save({
    'best_model': best_model,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'y_mean': y_mean,
    'y_std': y_std,
    'test_idx': test_idx,
    'train_idx': train_idx,
    'val_idx': val_idx,
    'shape': shape,
    'transform': transform,
    'node_to_idx': node_to_idx,
}, 'canwell_sage_nobasin.pt')
print("Saved!")

Saved!


In [19]:
# infer and retrieve errors

test_loader = NeighborLoader(
    data,
    num_neighbors=[10, 10, 10],
    batch_size=512,
    input_nodes=test_idx,
    shuffle=False,
)

model.load_state_dict(best_model)
model.eval()
preds = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        out = out[:batch.batch_size]
        tgt = batch.y[:batch.batch_size]
        preds.append(out.cpu())
        targets.append(tgt.cpu())

preds_m   = torch.cat(preds).numpy() * y_std + y_mean
targets_m = torch.cat(targets).numpy() * y_std + y_mean

mae  = np.mean(np.abs(preds_m - targets_m))
rmse = np.sqrt(np.mean((preds_m - targets_m)**2))
r2   = 1 - np.sum((targets_m - preds_m)**2) / np.sum((targets_m - np.mean(targets_m))**2)

print(f"NO BASIN - Test RMSE: {rmse:.3f} m")
print(f"NO BASIN - Test MAE:  {mae:.3f} m")
print(f"NO BASIN - Test R²:   {r2:.4f}")

NO BASIN - Test RMSE: 7.569 m
NO BASIN - Test MAE:  4.590 m
NO BASIN - Test R²:   0.7635


In [20]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 39.3%


In [27]:
feat = torch.load('../data/canwell_features.pt')
node_to_idx = feat['node_to_idx']
del feat
import gc
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 95.6%


In [28]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 95.5%


import pickle
with open('../data/canwell_graph.pkl', 'rb') as f:
    graph_data = pickle.load(f)

shape = graph_data['shape']
transform = graph_data['transform']

idx_to_demidx = {}
for node_id, attr in graph_data['G'].nodes(data=True):
    idx = node_to_idx[node_id]
    idx_to_demidx[idx] = attr['dem_idx']

del graph_data['G']
import gc
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

In [30]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 43.7%


In [32]:
print(len(idx_to_demidx))

4792230


In [33]:
del graph_data
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 43.1%


In [34]:
%who

CanwellSAGE	 Data	 F	 Linear	 NeighborLoader	 SAGEConv	 StandardScaler	 attr	 batch	 
best_epoch	 best_model	 best_val_loss	 criterion	 data	 device	 diff_context	 edge_index	 epoch	 
f	 gc	 has_diff	 idx	 idx_to_demidx	 is_basin	 is_masked	 is_slope	 mae	 
max_epochs	 model	 n	 no_improve	 node_id	 node_to_idx	 np	 optimizer	 out	 
patience	 perm	 pickle	 preds	 preds_m	 psutil	 r2	 rmse	 scaler	 
shape	 slope_with_diff	 targets	 targets_m	 test_idx	 test_loader	 tgt	 torch	 train	 
train_end	 train_idx	 train_loader	 train_loss	 train_losses	 transform	 val_end	 val_idx	 val_loader	 
val_loss	 val_losses	 validate	 x	 x_final	 x_np	 x_scaled	 y	 y_mean	 
y_norm	 y_std	 


In [35]:
del data, edge_index, x, x_final, x_np, x_scaled, y, y_norm
del train_loader, val_loader, test_loader
del preds, targets, preds_m, targets_m
del is_masked, diff_context, is_basin
del scaler, slope_with_diff, perm
del train_losses, val_losses
del optimizer, criterion
del batch, out, tgt
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 35.8%


In [38]:
# full slope inference
all_slope_idx = (is_slope & has_diff).nonzero(as_tuple=True)[0]

all_slope_loader = NeighborLoader(
    data, num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=all_slope_idx,
    shuffle=False,
)

model.eval()
all_preds = []
with torch.no_grad():
    for batch in all_slope_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        all_preds.append(out[:batch.batch_size].cpu())

all_preds_m = torch.cat(all_preds).numpy() * y_std + y_mean
print(f"Predictions: {len(all_preds_m):,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

NameError: name 'data' is not defined

In [37]:
import rasterio
import numpy as np

meta = {
    'driver': 'GTiff',
    'dtype': 'float32',
    'width': shape[1],
    'height': shape[0],
    'count': 1,
    'crs': data_reload['crs'] if 'crs' in dir() else rasterio.open('../data/Canwell_FullDomain_DEM3m.tif').crs,
    'transform': transform,
    'nodata': float('nan')
}

recon_nobasin = np.full(shape, np.nan, dtype=np.float32)
for i, node_id in enumerate(all_slope_idx.numpy()):
    r, c = idx_to_demidx[node_id]
    recon_nobasin[r, c] = all_preds_m[i]

with rasterio.open('canwell_gnn_differencenobasin.tif', 'w', **meta) as dst:
    dst.write(recon_nobasin, 1)
print(f"Saved. Filled: {np.sum(~np.isnan(recon_nobasin)):,} pixels")

NameError: name 'all_slope_idx' is not defined